# T1 mouse brain-extraction benchmark

This notebook runs the same uploaded pre-Gd T1 images through three open-weight predictions:

1. **MouseBrainExtractor `invivo_iso`** — the rule-based primary MBE configuration for this cohort.
2. **MouseBrainExtractor `invivo_aniso`** — a sensitivity run; the current anisotropy ratio is about 1.9, below the authors' >3 rule.
3. **RS2-Net** — an independently developed rodent skull-stripping network.

The official BEN release is not included: it requires legacy TensorFlow 1.15/Keras 2.2.4, and the official repository does not yet provide the advertised modern nnBEN code or weights. A failing or improvised BEN cell would not be a reproducible comparison.

## Exact use

1. In Colab choose **Runtime → Change runtime type → T4 GPU**.
2. Choose **Runtime → Run all**.
3. When prompted, upload `t1_brain_extraction_benchmark_10.zip`.
4. Keep this browser tab open. Inference can take a long time because all three predictions are 3D/2D deep networks.
5. The final cell downloads `t1_brain_extraction_results.zip`.

All masks are automatic pre-labels. Visual review remains mandatory before choosing a production model.

In [ ]:
# Configuration: these defaults run the complete quality-oriented benchmark.
from pathlib import Path
import os, shutil, subprocess, sys
import torch

RUN_MBE_ISO = True
RUN_MBE_ANISO = True
RUN_RS2NET = True
RS2_USE_TTA = True  # Official quality default; False is faster but less accurate.
MBE_BATCH_ROIS = 1  # Lowest-memory setting for Colab T4.

MBE_COMMIT = '5d67d622a0f67031494cc6e94867feabafee0bb8'
RS2_COMMIT = '144b032df4885a3da00e0d1824fdd777b3cd304f'
MBE_REPOSITORY = 'https://github.com/MouseSuite/MouseBrainExtractor.git'
RS2_REPOSITORY = 'https://github.com/VitoLin21/Rodent-Skull-Stripping.git'
MBE_WEIGHTS_URL = 'https://data.mousesuite.org/mbe/MBE_weights.tar.gz'
RS2_DRIVE_FOLDER_ID = '1cTlFFGL9iTUoZOT5Rgqi2ZAyqyPlXYd-'

BASE = Path('/content/lys_brain_benchmark')
PACKAGE_AREA = BASE / 'package'
EXTERNAL = BASE / 'external'
WORK = BASE / 'work'
RESULTS = BASE / 't1_brain_extraction_results'
for directory in (PACKAGE_AREA, EXTERNAL, WORK, RESULTS):
    directory.mkdir(parents=True, exist_ok=True)

if not torch.cuda.is_available():
    raise RuntimeError('No GPU detected. In Colab select Runtime → Change runtime type → T4 GPU, then rerun.')
print('GPU:', torch.cuda.get_device_name(0))
print('Python:', sys.version.split()[0], 'PyTorch:', torch.__version__)

In [ ]:
# Upload and unpack the locally prepared 10-image package.
from google.colab import files
import csv, json, zipfile

uploaded = files.upload()
zip_names = [name for name in uploaded if name.lower().endswith('.zip')]
if len(zip_names) != 1:
    raise ValueError(f'Upload exactly one .zip package; received: {list(uploaded)}')
upload_path = Path('/content') / zip_names[0]
if PACKAGE_AREA.exists():
    shutil.rmtree(PACKAGE_AREA)
PACKAGE_AREA.mkdir(parents=True)
with zipfile.ZipFile(upload_path) as archive:
    for member in archive.infolist():
        target = (PACKAGE_AREA / member.filename).resolve()
        if not target.is_relative_to(PACKAGE_AREA.resolve()):
            raise ValueError(f'Unsafe archive path: {member.filename}')
    archive.extractall(PACKAGE_AREA)
manifests = list(PACKAGE_AREA.rglob('benchmark_manifest.csv'))
if len(manifests) != 1:
    raise ValueError(f'Expected one benchmark_manifest.csv, found {len(manifests)}')
PACKAGE_ROOT = manifests[0].parent
with manifests[0].open(newline='') as stream:
    PACKAGE_ROWS = list(csv.DictReader(stream))
if len(PACKAGE_ROWS) != 10:
    raise ValueError(f'This benchmark expects exactly 10 cases, found {len(PACKAGE_ROWS)}')

if RESULTS.exists():
    shutil.rmtree(RESULTS)
(RESULTS / 'inputs').mkdir(parents=True)
CASES = []
for row in PACKAGE_ROWS:
    source = PACKAGE_ROOT / row['image']
    destination = RESULTS / 'inputs' / f"{row['case_id']}_pre_t1.nii.gz"
    shutil.copy2(source, destination)
    CASES.append({'case_id': row['case_id'], 'image': destination})
shutil.copy2(manifests[0], RESULTS / 'input_benchmark_manifest.csv')
if (PACKAGE_ROOT / 'package_metadata.json').is_file():
    shutil.copy2(PACKAGE_ROOT / 'package_metadata.json', RESULTS / 'input_package_metadata.json')
print('Package:', PACKAGE_ROOT)
print('Cases:', ', '.join(row['case_id'] for row in CASES))

In [ ]:
# Install inference dependencies, clone pinned source revisions, and download official weights.
import tarfile, urllib.request

packages = [
    'monai==1.4.0', 'nibabel>=5.3,<6', 'nilearn>=0.12,<1',
    'scikit-image>=0.23,<1', 'einops>=0.8,<1', 'gdown>=5.2,<6',
    'acvl_utils==0.2.1', 'batchgenerators>=0.25,<1',
    'SimpleITK>=2.4,<3', 'tifffile', 'imageio', 'pandas'
]
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '--upgrade', *packages])
runtime_preflight = (
    "import monai, torch; "
    "from monai.inferers import sliding_window_inference, SliceInferer; "
    "assert monai.__version__.startswith('1.4.'), monai.__version__; "
    "assert torch.cuda.is_available(), 'GPU unavailable to subprocess'; "
    "print('Runtime preflight: MONAI', monai.__version__, '| GPU', torch.cuda.get_device_name(0))"
)
subprocess.check_call([sys.executable, '-c', runtime_preflight])

def clone_pinned(repository, commit, destination):
    if destination.exists():
        shutil.rmtree(destination)
    subprocess.check_call(['git', 'clone', '-q', repository, str(destination)])
    subprocess.check_call(['git', '-C', str(destination), 'checkout', '-q', commit])
    actual = subprocess.check_output(['git', '-C', str(destination), 'rev-parse', 'HEAD'], text=True).strip()
    if actual != commit:
        raise RuntimeError(f'Pinned checkout failed: expected {commit}, got {actual}')

MBE_ROOT = EXTERNAL / 'MouseBrainExtractor'
RS2_ROOT = EXTERNAL / 'Rodent-Skull-Stripping'
clone_pinned(MBE_REPOSITORY, MBE_COMMIT, MBE_ROOT)
clone_pinned(RS2_REPOSITORY, RS2_COMMIT, RS2_ROOT)
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '-e', str(RS2_ROOT), '--no-deps'])

# RS2's official legacy checkpoint contains NumPy metadata. PyTorch >=2.6 changed
# torch.load to weights_only=True, so make the trusted official-checkpoint load explicit.
RS2_PREDICT_SOURCE = RS2_ROOT / 'RS2' / 'inference' / 'predict.py'
rs2_source = RS2_PREDICT_SOURCE.read_text()
old_load = "torch.load(checkpoint_name, map_location=torch.device('cpu'))"
new_load = "torch.load(checkpoint_name, map_location=torch.device('cpu'), weights_only=False)"
if new_load not in rs2_source:
    if rs2_source.count(old_load) != 1:
        raise RuntimeError('Could not apply the pinned RS2 PyTorch 2.6 checkpoint compatibility patch')
    RS2_PREDICT_SOURCE.write_text(rs2_source.replace(old_load, new_load))
RS2_CHECKPOINT_COMPATIBILITY = 'Explicit weights_only=False for the official RS2 legacy checkpoint under PyTorch >=2.6'

# Fail once, before large weight downloads or 30 inference attempts, if model APIs drift.
model_preflight = """
from models.swin_unetr_mod5 import SwinUNETR
from RS2.network.RSSNet import RSSNet
mbe = SwinUNETR(img_size=(96, 96, 96), in_channels=1, out_channels=2, feature_size=48, use_checkpoint=True, spatial_dims=3)
del mbe
rs2 = RSSNet(img_size=(128, 128, 160), in_channels=1, out_channels=1, feature_size=48)
del rs2
print('Model preflight: MouseBrainExtractor and RS2-Net constructors passed')
"""
preflight_env = os.environ.copy()
preflight_env['PYTHONPATH'] = os.pathsep.join([str(MBE_ROOT), str(RS2_ROOT), preflight_env.get('PYTHONPATH', '')])
subprocess.check_call([sys.executable, '-c', model_preflight], env=preflight_env)

MBE_WEIGHTS_AREA = EXTERNAL / 'mbe_weights'
mbe_tar = EXTERNAL / 'MBE_weights.tar.gz'
if not mbe_tar.is_file():
    print('Downloading MouseBrainExtractor weights (~531 MB)...')
    urllib.request.urlretrieve(MBE_WEIGHTS_URL, mbe_tar)
if not list(MBE_WEIGHTS_AREA.rglob('checkpoint_best.pth')):
    MBE_WEIGHTS_AREA.mkdir(parents=True, exist_ok=True)
    with tarfile.open(mbe_tar) as archive:
        for member in archive.getmembers():
            target = (MBE_WEIGHTS_AREA / member.name).resolve()
            if not target.is_relative_to(MBE_WEIGHTS_AREA.resolve()):
                raise ValueError(f'Unsafe weights archive path: {member.name}')
        archive.extractall(MBE_WEIGHTS_AREA)

import gdown
RS2_WEIGHTS_AREA = EXTERNAL / 'rs2_weights'
RS2_WEIGHTS_AREA.mkdir(parents=True, exist_ok=True)
if not list(RS2_WEIGHTS_AREA.rglob('*pretrained_model.pt')):
    print('Downloading RS2-Net official pretrained weights...')
    gdown.download_folder(
        id=RS2_DRIVE_FOLDER_ID, output=str(RS2_WEIGHTS_AREA), quiet=False, use_cookies=False
    )
rs2_downloaded = list(RS2_WEIGHTS_AREA.rglob('*pretrained_model.pt'))
if len(rs2_downloaded) != 1:
    raise FileNotFoundError(f'Expected one official RS2 checkpoint, found {rs2_downloaded}')
checkpoint_preflight = (
    "import torch; "
    f"checkpoint=torch.load({str(rs2_downloaded[0])!r}, map_location='cpu', weights_only=False); "
    "assert isinstance(checkpoint, dict) and 'state_dict' in checkpoint; "
    "print('RS2 checkpoint preflight: state_dict found')"
)
subprocess.check_call([sys.executable, '-c', checkpoint_preflight])
print('Pinned repositories and official weights are ready.')

In [ ]:
# Shared output contract, logging, provenance, and NIfTI standardization helpers.
import csv, hashlib, importlib.metadata, json, platform, time
from datetime import datetime, timezone
import nibabel as nib
from nibabel.processing import resample_from_to
import numpy as np

RUN_FIELDS = ['case_id', 'model_id', 'status', 'image', 'mask', 'metadata', 'log', 'message']
RUN_ROWS = {}

def utc_now():
    return datetime.now(timezone.utc).isoformat(timespec='seconds')

def sha256(path, chunk_size=1024 * 1024):
    digest = hashlib.sha256()
    with Path(path).open('rb') as stream:
        while chunk := stream.read(chunk_size):
            digest.update(chunk)
    return digest.hexdigest()

def relative(path):
    return str(Path(path).resolve().relative_to(RESULTS.resolve())) if path else ''

def write_run_manifest():
    path = RESULTS / 'run_manifest.csv'
    with path.open('w', newline='') as stream:
        writer = csv.DictWriter(stream, fieldnames=RUN_FIELDS)
        writer.writeheader()
        writer.writerows(RUN_ROWS[key] for key in sorted(RUN_ROWS))

def record(case_id, model_id, status, image, mask=None, metadata=None, log=None, message=''):
    RUN_ROWS[(case_id, model_id)] = {
        'case_id': case_id, 'model_id': model_id, 'status': status,
        'image': relative(image), 'mask': relative(mask),
        'metadata': relative(metadata), 'log': relative(log), 'message': message,
    }
    write_run_manifest()

def run_logged(command, log_path, cwd=None):
    log_path.parent.mkdir(parents=True, exist_ok=True)
    print('$', ' '.join(str(part) for part in command))
    with log_path.open('w') as log:
        process = subprocess.Popen(
            [str(part) for part in command], cwd=cwd, stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT, text=True, bufsize=1
        )
        assert process.stdout is not None
        for line in process.stdout:
            print(line, end='')
            log.write(line)
        return process.wait()

def find_mbe_checkpoint(dataset_type):
    candidates = [
        path for path in MBE_WEIGHTS_AREA.rglob('checkpoint_best.pth')
        if path.parent.name == dataset_type
    ]
    if len(candidates) != 1:
        raise FileNotFoundError(f'Expected one {dataset_type} checkpoint, found {candidates}')
    return candidates[0]

rs2_candidates = list(RS2_WEIGHTS_AREA.rglob('*pretrained_model.pt'))
if len(rs2_candidates) != 1:
    raise FileNotFoundError(f'Expected one RS2 pretrained checkpoint, found {rs2_candidates}')
MBE_ISO_WEIGHT = find_mbe_checkpoint('invivo_iso')
MBE_ANISO_WEIGHT = find_mbe_checkpoint('invivo_aniso')
RS2_WEIGHT = rs2_candidates[0]
WEIGHT_HASHES = {
    'mbe_invivo_iso': sha256(MBE_ISO_WEIGHT),
    'mbe_invivo_aniso': sha256(MBE_ANISO_WEIGHT),
    'rs2net': sha256(RS2_WEIGHT),
}

def standardize_mask(source, reference, destination):
    reference_image = nib.load(str(reference))
    mask_image = nib.load(str(source))
    resampled = False
    if mask_image.shape != reference_image.shape or not np.allclose(mask_image.affine, reference_image.affine, rtol=1e-5, atol=1e-5):
        mask_image = resample_from_to(mask_image, reference_image, order=0)
        resampled = True
    mask = (np.asanyarray(mask_image.dataobj) > 0).astype(np.uint8)
    if not mask.any() or mask.all():
        raise ValueError(f'Invalid constant mask after standardization: foreground={int(mask.sum())}')
    header = reference_image.header.copy()
    header.set_data_dtype(np.uint8)
    destination.parent.mkdir(parents=True, exist_ok=True)
    output = nib.Nifti1Image(mask, reference_image.affine, header)
    output.set_qform(reference_image.affine, code=int(reference_image.header['qform_code']))
    output.set_sform(reference_image.affine, code=int(reference_image.header['sform_code']))
    nib.save(output, destination)
    return {'resampled_to_input_grid': resampled, 'foreground_voxels': int(mask.sum())}

RUNTIME = {
    'created_at': utc_now(), 'python': platform.python_version(),
    'pytorch': torch.__version__, 'monai': importlib.metadata.version('monai'),
    'cuda': torch.version.cuda,
    'gpu': torch.cuda.get_device_name(0),
}
MODEL_PROVENANCE = {
    'mbe_invivo_iso': {'repository': MBE_REPOSITORY, 'commit': MBE_COMMIT, 'weights_url': MBE_WEIGHTS_URL, 'weight_sha256': WEIGHT_HASHES['mbe_invivo_iso']},
    'mbe_invivo_aniso': {'repository': MBE_REPOSITORY, 'commit': MBE_COMMIT, 'weights_url': MBE_WEIGHTS_URL, 'weight_sha256': WEIGHT_HASHES['mbe_invivo_aniso']},
    'rs2net': {'repository': RS2_REPOSITORY, 'commit': RS2_COMMIT, 'weights_url': f'https://drive.google.com/drive/folders/{RS2_DRIVE_FOLDER_ID}', 'weight_sha256': WEIGHT_HASHES['rs2net'], 'checkpoint_compatibility': RS2_CHECKPOINT_COMPATIBILITY, 'tta': RS2_USE_TTA},
}
print(json.dumps({'runtime': RUNTIME, 'models': MODEL_PROVENANCE}, indent=2))

In [ ]:
# MouseBrainExtractor runner used by both official in-vivo configurations.
def run_mbe_model(model_id, dataset_type, weight, dimensions, patch_size):
    script = MBE_ROOT / 'bin' / 'run_mbe_predict_skullstrip.py'
    for index, case in enumerate(CASES, start=1):
        case_id, image = case['case_id'], case['image']
        print(f'\n[{index}/{len(CASES)}] {model_id}: {case_id}')
        raw_dir = WORK / model_id / case_id
        raw_dir.mkdir(parents=True, exist_ok=True)
        raw = raw_dir / f'{case_id}_raw.nii.gz'
        posenc = raw_dir / f'{case_id}_posenc.nii.gz'
        mask = RESULTS / 'predictions' / model_id / f'{case_id}_brain_mask.nii.gz'
        metadata = RESULTS / 'metadata' / model_id / f'{case_id}.json'
        log = RESULTS / 'logs' / model_id / f'{case_id}.log'
        metadata.parent.mkdir(parents=True, exist_ok=True)
        command = [
            sys.executable, str(script), '-i', str(image), '--gen_posenc', str(posenc),
            '-o', str(raw), '-n', str(weight), '--device', 'cuda', '--pp',
            '--dstype', 'invivo', '-b', str(MBE_BATCH_ROIS),
        ]
        if dimensions == 2:
            command.extend(['-d', '2', '--patch_size', str(patch_size)])
        started = time.monotonic()
        try:
            returncode = run_logged(command, log, cwd=MBE_ROOT)
            if returncode != 0:
                raise RuntimeError(f'MouseBrainExtractor exited with status {returncode}')
            pp = Path(str(raw).split('.nii')[0] + '.pp.nii.gz')
            source = pp if pp.is_file() else raw
            standard = standardize_mask(source, image, mask)
            payload = {
                'case_id': case_id, 'model_id': model_id, 'status': 'ok',
                'input': relative(image), 'output': relative(mask),
                'official_dataset_type': dataset_type, 'dimensions': dimensions,
                'patch_size': patch_size,
                'preprocessing': 'official NormalizeIntensityd(nonzero=True)',
                'official_postprocessing': 'hole filling, largest component, and model-specific smoothing',
                'standardization_threshold': 0,
                'command': [str(part) for part in command],
                'runtime_seconds': round(time.monotonic() - started, 3),
                'input_sha256': sha256(image), 'mask_sha256': sha256(mask),
                **standard, **MODEL_PROVENANCE[model_id], **RUNTIME,
            }
            metadata.write_text(json.dumps(payload, indent=2) + '\n')
            record(case_id, model_id, 'ok', image, mask, metadata, log)
        except Exception as exc:
            message = f'{type(exc).__name__}: {exc}'
            log.parent.mkdir(parents=True, exist_ok=True)
            with log.open('a') as stream:
                stream.write('\nBENCHMARK ERROR: ' + message + '\n')
            record(case_id, model_id, 'failed', image, log=log, message=message)
            print('FAILED:', message)
        finally:
            if torch.cuda.is_available():
                torch.cuda.empty_cache()

if RUN_MBE_ISO:
    run_mbe_model('mbe_invivo_iso', 'invivo_iso', MBE_ISO_WEIGHT, 3, 96)
else:
    print('Skipped mbe_invivo_iso by configuration.')

In [ ]:
# MouseBrainExtractor anisotropic sensitivity run.
if RUN_MBE_ANISO:
    run_mbe_model('mbe_invivo_aniso', 'invivo_aniso', MBE_ANISO_WEIGHT, 2, 128)
else:
    print('Skipped mbe_invivo_aniso by configuration.')

In [ ]:
# RS2-Net batch prediction using its official filename and inference interface.
if RUN_RS2NET:
    model_id = 'rs2net'
    rs2_input = WORK / 'rs2net_inputs'
    rs2_raw = WORK / 'rs2net_outputs'
    for directory in (rs2_input, rs2_raw):
        if directory.exists():
            shutil.rmtree(directory)
        directory.mkdir(parents=True)
    for case in CASES:
        shutil.copy2(case['image'], rs2_input / f"{case['case_id']}_0000.nii.gz")
    batch_log = RESULTS / 'logs' / model_id / 'rs2net_batch.log'
    command = [
        'RS2_predict', '-i', str(rs2_input), '-o', str(rs2_raw),
        '-m', str(RS2_WEIGHT), '-device', 'cuda', '-npp', '1', '-nps', '1',
    ]
    if not RS2_USE_TTA:
        command.append('--disable_tta')
    started = time.monotonic()
    returncode = run_logged(command, batch_log, cwd=RS2_ROOT)
    elapsed = round(time.monotonic() - started, 3)
    for case in CASES:
        case_id, image = case['case_id'], case['image']
        mask = RESULTS / 'predictions' / model_id / f'{case_id}_brain_mask.nii.gz'
        metadata = RESULTS / 'metadata' / model_id / f'{case_id}.json'
        metadata.parent.mkdir(parents=True, exist_ok=True)
        candidates = [rs2_raw / f'{case_id}.nii.gz', rs2_raw / f'{case_id}_0000.nii.gz']
        source = next((path for path in candidates if path.is_file()), None)
        try:
            if source is None:
                raise FileNotFoundError(f'RS2-Net did not create an output; batch status {returncode}')
            standard = standardize_mask(source, image, mask)
            payload = {
                'case_id': case_id, 'model_id': model_id, 'status': 'ok',
                'input': relative(image), 'output': relative(mask),
                'official_input_suffix': '_0000.nii.gz',
                'preprocessing': 'official RS2 DefaultPreprocessor',
                'standardization_threshold': 0, 'test_time_augmentation': RS2_USE_TTA,
                'batch_command': command, 'batch_runtime_seconds': elapsed,
                'input_sha256': sha256(image), 'mask_sha256': sha256(mask),
                **standard, **MODEL_PROVENANCE[model_id], **RUNTIME,
            }
            metadata.write_text(json.dumps(payload, indent=2) + '\n')
            record(case_id, model_id, 'ok', image, mask, metadata, batch_log)
        except Exception as exc:
            message = f'{type(exc).__name__}: {exc}'
            record(case_id, model_id, 'failed', image, log=batch_log, message=message)
            print(f'FAILED {case_id}: {message}')
else:
    print('Skipped rs2net by configuration.')

In [ ]:
# OPTIONAL: RS2-Net probability threshold sweep. Run after the standard RS2 cell.
# It reruns RS2 only once, then creates six masks without rerunning inference.
from itertools import permutations
import ipywidgets as widgets
import matplotlib.pyplot as plt
from IPython.display import display

RS2_THRESHOLDS = (0.50, 0.60, 0.70, 0.80, 0.90, 0.95)
SWEEP_ROOT = BASE / 'rs2_threshold_sweep_results'
SWEEP_INPUT = WORK / 'rs2net_probability_inputs'
SWEEP_RAW = WORK / 'rs2net_probability_outputs'
for directory in (SWEEP_ROOT, SWEEP_INPUT):
    if directory.exists():
        shutil.rmtree(directory)
    directory.mkdir(parents=True)
SWEEP_RAW.mkdir(parents=True, exist_ok=True)
(SWEEP_ROOT / 'inputs').mkdir(parents=True)

for case in CASES:
    shutil.copy2(case['image'], SWEEP_INPUT / f"{case['case_id']}_0000.nii.gz")
    shutil.copy2(case['image'], SWEEP_ROOT / 'inputs' / f"{case['case_id']}_pre_t1.nii.gz")

sweep_log = SWEEP_ROOT / 'logs' / 'rs2net_probability_batch.log'
sweep_command = [
    'RS2_predict', '-i', str(SWEEP_INPUT), '-o', str(SWEEP_RAW),
    '-m', str(RS2_WEIGHT), '-device', 'cuda', '-npp', '1', '-nps', '1',
    '--save_probabilities',
]
if not RS2_USE_TTA:
    sweep_command.append('--disable_tta')
outputs_ready = all(
    len(list(SWEEP_RAW.glob(f"{case['case_id']}*.npz"))) == 1
    and any((SWEEP_RAW / name).is_file() for name in (
        f"{case['case_id']}_0000.nii.gz", f"{case['case_id']}.nii.gz"
    ))
    for case in CASES
)
if outputs_ready:
    sweep_runtime = None
    sweep_log.parent.mkdir(parents=True, exist_ok=True)
    sweep_log.write_text('Reused complete RS2 probability outputs from the current Colab runtime.\n')
    print('Reusing the 10 complete RS2 probability outputs already present in this runtime.')
else:
    if SWEEP_RAW.exists():
        shutil.rmtree(SWEEP_RAW)
    SWEEP_RAW.mkdir(parents=True)
    sweep_started = time.monotonic()
    sweep_status = run_logged(sweep_command, sweep_log, cwd=RS2_ROOT)
    if sweep_status != 0:
        raise RuntimeError(f'RS2 probability inference exited with status {sweep_status}')
    sweep_runtime = round(time.monotonic() - sweep_started, 3)

def save_native_nifti(array, reference, destination, dtype):
    destination.parent.mkdir(parents=True, exist_ok=True)
    header = reference.header.copy()
    header.set_data_dtype(dtype)
    output = nib.Nifti1Image(np.asarray(array, dtype=dtype), reference.affine, header)
    output.set_qform(reference.affine, code=int(reference.header['qform_code']))
    output.set_sform(reference.affine, code=int(reference.header['sform_code']))
    nib.save(output, destination)

def dice_score(first, second):
    denominator = int(first.sum()) + int(second.sum())
    return 1.0 if denominator == 0 else 2.0 * int(np.logical_and(first, second).sum()) / denominator

sweep_rows = []
SWEEP_CACHE = {}
for case in CASES:
    case_id, input_image = case['case_id'], case['image']
    print('Preparing thresholds:', case_id)
    probability_candidates = sorted(SWEEP_RAW.glob(f'{case_id}*.npz'))
    official_candidates = [SWEEP_RAW / f'{case_id}_0000.nii.gz', SWEEP_RAW / f'{case_id}.nii.gz']
    official_path = next((path for path in official_candidates if path.is_file()), None)
    if len(probability_candidates) != 1 or official_path is None:
        raise FileNotFoundError(
            f'Expected one probability file and one official mask for {case_id}; '
            f'probabilities={probability_candidates}, masks={official_candidates}'
        )
    with np.load(probability_candidates[0]) as payload:
        if 'probabilities' not in payload:
            raise KeyError(f"Missing 'probabilities' in {probability_candidates[0]}: {payload.files}")
        raw_probability = np.asarray(payload['probabilities'], dtype=np.float32)
    if raw_probability.ndim == 3:
        raw_probability = raw_probability[None]
    if raw_probability.ndim != 4:
        raise ValueError(f'Unexpected channel-first probability shape for {case_id}: {raw_probability.shape}')
    if not np.isfinite(raw_probability).all() or raw_probability.min() < 0 or raw_probability.max() > 1:
        raise ValueError(f'Invalid probability range for {case_id}: {raw_probability.min()}..{raw_probability.max()}')

    reference = nib.load(str(input_image))
    official = np.asanyarray(nib.load(str(official_path)).dataobj) > 0
    orientation_candidates = []
    # The pinned RS2 LabelManager defines its discrete segmentation as
    # sigmoid(channel 0) > 0.5. Test every exported channel defensively,
    # then select the one that reproduces the official NIfTI mask.
    source_candidates = [
        (channel, raw_probability[channel] > 0.50)
        for channel in range(raw_probability.shape[0])
    ]
    for foreground_channel, source_mask in source_candidates:
        for axis_order in permutations(range(3)):
            candidate = np.transpose(source_mask, axis_order)
            if candidate.shape == reference.shape:
                orientation_candidates.append((
                    dice_score(candidate, official), axis_order, foreground_channel
                ))
    if not orientation_candidates:
        raise ValueError(
            f'No probability permutation matches the input grid for {case_id}: '
            f'{raw_probability.shape} versus {reference.shape}'
        )
    orientation_dice, axis_order, foreground_channel = max(orientation_candidates, key=lambda item: item[0])
    if orientation_dice < 0.99:
        raise ValueError(
            f'Probability orientation/channel check failed for {case_id}: '
            f'Dice={orientation_dice:.5f}, axes={axis_order}, foreground={foreground_channel}'
        )
    probability = np.transpose(raw_probability[foreground_channel], axis_order).astype(np.float32)
    probability_conversion = 'official exported foreground sigmoid channel selected against discrete mask'
    reconstructed_dice = dice_score(probability > 0.50, official)
    if reconstructed_dice < 0.99:
        raise ValueError(
            f'Reconstructed 0.50 probability mask disagrees with official output for {case_id}: '
            f'Dice={reconstructed_dice:.5f}'
        )

    probability_path = SWEEP_ROOT / 'probabilities' / f'{case_id}_rs2net_probability.nii.gz'
    save_native_nifti(probability, reference, probability_path, np.float32)
    threshold_masks = {}
    for threshold in RS2_THRESHOLDS:
        suffix = f'p{int(round(threshold * 100)):03d}'
        model_id = f'rs2net_{suffix}'
        mask_data = (probability > threshold).astype(np.uint8)
        if not mask_data.any() or mask_data.all():
            raise ValueError(f'Constant {threshold:.2f} mask for {case_id}')
        mask_path = SWEEP_ROOT / 'predictions' / model_id / f'{case_id}_brain_mask.nii.gz'
        metadata_path = SWEEP_ROOT / 'metadata' / model_id / f'{case_id}.json'
        save_native_nifti(mask_data, reference, mask_path, np.uint8)
        metadata_path.parent.mkdir(parents=True, exist_ok=True)
        metadata = {
            'case_id': case_id, 'model_id': model_id, 'status': 'ok',
            'threshold': threshold, 'source_probability': str(probability_path.relative_to(SWEEP_ROOT)),
            'probability_axis_permutation': list(axis_order),
            'foreground_channel': int(foreground_channel),
            'probability_conversion': probability_conversion,
            'orientation_validation_dice': orientation_dice,
            'orientation_validation_dice_at_0.50': reconstructed_dice,
            'foreground_voxels': int(mask_data.sum()),
            'input_sha256': sha256(input_image), 'mask_sha256': sha256(mask_path),
            'batch_runtime_seconds': sweep_runtime, 'batch_command': sweep_command,
            **MODEL_PROVENANCE['rs2net'], **RUNTIME,
        }
        metadata_path.write_text(json.dumps(metadata, indent=2) + '\n')
        threshold_masks[threshold] = mask_data.astype(bool)
        sweep_rows.append({
            'case_id': case_id, 'model_id': model_id, 'status': 'ok',
            'image': f'inputs/{case_id}_pre_t1.nii.gz',
            'mask': str(mask_path.relative_to(SWEEP_ROOT)),
            'metadata': str(metadata_path.relative_to(SWEEP_ROOT)),
            'log': str(sweep_log.relative_to(SWEEP_ROOT)), 'message': '',
        })
    SWEEP_CACHE[case_id] = {
        'image': reference.get_fdata(dtype=np.float32),
        'masks': threshold_masks,
    }

with (SWEEP_ROOT / 'run_manifest.csv').open('w', newline='') as stream:
    writer = csv.DictWriter(stream, fieldnames=RUN_FIELDS)
    writer.writeheader(); writer.writerows(sweep_rows)
(SWEEP_ROOT / 'run_metadata.json').write_text(json.dumps({
    'purpose': 'RS2-Net probability-threshold calibration',
    'thresholds': list(RS2_THRESHOLDS), 'case_count': len(CASES),
    'warning': 'Thresholds are candidates, not approved masks. Select using all reviewed cases.',
    'model_provenance': MODEL_PROVENANCE['rs2net'], 'runtime': RUNTIME,
}, indent=2) + '\n')

# Save one six-threshold x six-slice montage for every case.
qc_dir = SWEEP_ROOT / 'qc'
qc_dir.mkdir(parents=True, exist_ok=True)
for case_id, cached in SWEEP_CACHE.items():
    image_data = cached['image']
    nonzero = image_data[np.isfinite(image_data) & (image_data != 0)]
    vmin, vmax = np.percentile(nonzero, [1, 99.5]) if nonzero.size else (0, 1)
    start, stop = min(50, image_data.shape[2] - 1), min(170, image_data.shape[2] - 1)
    slices = np.linspace(start, stop, 6).round().astype(int)
    fig, axes = plt.subplots(len(RS2_THRESHOLDS), len(slices), figsize=(16, 14), squeeze=False)
    for row_index, threshold in enumerate(RS2_THRESHOLDS):
        for column_index, slice_index in enumerate(slices):
            axis = axes[row_index, column_index]
            axis.imshow(np.rot90(image_data[:, :, slice_index]), cmap='gray', vmin=vmin, vmax=vmax)
            axis.contour(np.rot90(cached['masks'][threshold][:, :, slice_index]), levels=[0.5], colors='lime', linewidths=0.8)
            axis.set_xticks([]); axis.set_yticks([])
            if row_index == 0:
                axis.set_title(f'slice {slice_index}', fontsize=9)
            if column_index == 0:
                axis.set_ylabel(f'p >= {threshold:.2f}', fontsize=9)
    fig.suptitle(f'{case_id}: RS2-Net probability thresholds', fontsize=13)
    fig.tight_layout()
    fig.savefig(qc_dir / f'{case_id}_rs2_thresholds.png', dpi=150, bbox_inches='tight')
    plt.close(fig)

# Interactive in-Colab viewer: select a mouse and move through the displayed slices.
def show_rs2_thresholds(case_id, slice_index):
    cached = SWEEP_CACHE[case_id]
    image_data = cached['image']
    slice_index = min(int(slice_index), image_data.shape[2] - 1)
    nonzero = image_data[np.isfinite(image_data) & (image_data != 0)]
    vmin, vmax = np.percentile(nonzero, [1, 99.5]) if nonzero.size else (0, 1)
    fig, axes = plt.subplots(2, 3, figsize=(13, 8), squeeze=False)
    for axis, threshold in zip(axes.ravel(), RS2_THRESHOLDS):
        axis.imshow(np.rot90(image_data[:, :, slice_index]), cmap='gray', vmin=vmin, vmax=vmax)
        axis.contour(np.rot90(cached['masks'][threshold][:, :, slice_index]), levels=[0.5], colors='lime', linewidths=1.1)
        axis.set_title(f'RS2 probability >= {threshold:.2f}')
        axis.axis('off')
    fig.suptitle(f'{case_id} — displayed slice {slice_index}')
    fig.tight_layout(); plt.show()

case_widget = widgets.Dropdown(options=sorted(SWEEP_CACHE), description='Case:')
slice_widget = widgets.IntSlider(
    value=min(125, SWEEP_CACHE[sorted(SWEEP_CACHE)[0]]['image'].shape[2] - 1),
    min=0, max=max(cached['image'].shape[2] for cached in SWEEP_CACHE.values()) - 1,
    step=1, description='Slice:', continuous_update=False,
)
display(widgets.interactive_output(show_rs2_thresholds, {'case_id': case_widget, 'slice_index': slice_widget}))
display(widgets.HBox([case_widget, slice_widget]))

sweep_archive = Path('/content/rs2_threshold_sweep_results.zip')
if sweep_archive.exists():
    sweep_archive.unlink()
sweep_archive = Path(shutil.make_archive(
    str(sweep_archive.with_suffix('')), 'zip', root_dir=SWEEP_ROOT.parent, base_dir=SWEEP_ROOT.name
))
print(f'Created {len(sweep_rows)} threshold masks and {len(SWEEP_CACHE)} QC montages.')
print('Download:', sweep_archive)
files.download(str(sweep_archive))


In [ ]:
# Create one immediate visual comparison montage per case.
import matplotlib.pyplot as plt
from IPython.display import display, Image

MODEL_ORDER = ['mbe_invivo_iso', 'mbe_invivo_aniso', 'rs2net']
MODEL_TITLES = {
    'mbe_invivo_iso': 'MBE iso', 'mbe_invivo_aniso': 'MBE aniso', 'rs2net': 'RS2-Net'
}
qc_dir = RESULTS / 'qc'
qc_dir.mkdir(parents=True, exist_ok=True)
created_qc = []
for case in CASES:
    case_id, image_path = case['case_id'], case['image']
    available = [model for model in MODEL_ORDER if RUN_ROWS.get((case_id, model), {}).get('status') == 'ok']
    if not available:
        continue
    image_data = nib.load(str(image_path)).get_fdata(dtype=np.float32)
    nonzero = image_data[np.isfinite(image_data) & (image_data != 0)]
    vmin, vmax = np.percentile(nonzero, [1, 99.5]) if nonzero.size else (0, 1)
    start, stop = min(50, image_data.shape[2] - 1), min(170, image_data.shape[2] - 1)
    slices = np.linspace(start, stop, 6).round().astype(int)
    fig, axes = plt.subplots(len(available), len(slices), figsize=(15, 2.7 * len(available)), squeeze=False)
    for row_index, model_id in enumerate(available):
        mask_path = RESULTS / RUN_ROWS[(case_id, model_id)]['mask']
        mask = np.asanyarray(nib.load(str(mask_path)).dataobj) > 0
        for column_index, slice_index in enumerate(slices):
            axis = axes[row_index, column_index]
            axis.imshow(np.rot90(image_data[:, :, slice_index]), cmap='gray', vmin=vmin, vmax=vmax)
            axis.contour(np.rot90(mask[:, :, slice_index]), levels=[0.5], colors='lime', linewidths=0.8)
            axis.set_xticks([]); axis.set_yticks([])
            if row_index == 0:
                axis.set_title(f'slice {slice_index}', fontsize=9)
            if column_index == 0:
                axis.set_ylabel(MODEL_TITLES[model_id], fontsize=10)
    fig.suptitle(f'{case_id}: automatic brain-mask boundaries', fontsize=12)
    fig.tight_layout()
    output = qc_dir / f'{case_id}_model_comparison.png'
    fig.savefig(output, dpi=160, bbox_inches='tight')
    plt.close(fig)
    created_qc.append(output)
print(f'Created {len(created_qc)} QC montages.')
if created_qc:
    display(Image(filename=str(created_qc[0])))

In [ ]:
# Validate the standardized outputs, write final provenance, zip everything, and download it.
validation_rows = []
for key in sorted(RUN_ROWS):
    row = RUN_ROWS[key]
    validation = {'case_id': row['case_id'], 'model_id': row['model_id'], 'run_status': row['status'], 'grid_ok': '', 'binary_ok': '', 'message': row['message']}
    if row['status'] == 'ok':
        try:
            image = nib.load(str(RESULTS / row['image']))
            mask = nib.load(str(RESULTS / row['mask']))
            values = np.unique(np.asanyarray(mask.dataobj))
            validation['grid_ok'] = bool(image.shape == mask.shape and np.allclose(image.affine, mask.affine, rtol=1e-5, atol=1e-5))
            validation['binary_ok'] = bool(set(values.tolist()).issubset({0, 1}) and values.size > 1)
            if not validation['grid_ok'] or not validation['binary_ok']:
                validation['message'] = f'grid_ok={validation["grid_ok"]}, binary_ok={validation["binary_ok"]}, values={values[:8]}'
        except Exception as exc:
            validation['message'] = f'{type(exc).__name__}: {exc}'
    validation_rows.append(validation)
with (RESULTS / 'validation_summary.csv').open('w', newline='') as stream:
    writer = csv.DictWriter(stream, fieldnames=['case_id', 'model_id', 'run_status', 'grid_ok', 'binary_ok', 'message'])
    writer.writeheader(); writer.writerows(validation_rows)

run_metadata = {
    'purpose': 'Visual comparison of open-weight mouse/rodent T1 brain-extraction pre-labels',
    'created_at': utc_now(), 'case_count': len(CASES),
    'case_ids': [case['case_id'] for case in CASES],
    'model_provenance': MODEL_PROVENANCE, 'runtime': RUNTIME,
    'excluded_candidate': {
        'name': 'BEN/nnBEN',
        'reason': 'Official BEN requires TensorFlow 1.15/Keras 2.2.4; official nnBEN code and weights are not released.'
    },
    'warning': 'These are automatic pre-labels and require human review.'
}
(RESULTS / 'run_metadata.json').write_text(json.dumps(run_metadata, indent=2) + '\n')
write_run_manifest()

ok = sum(row['status'] == 'ok' for row in RUN_ROWS.values())
failed = sum(row['status'] != 'ok' for row in RUN_ROWS.values())
print(f'Predictions ready: {ok}; failed/skipped: {failed}; expected when all enabled: {len(CASES) * 3}')
for row in RUN_ROWS.values():
    if row['status'] != 'ok':
        print('FAILED:', row['case_id'], row['model_id'], row['message'])

archive_base = Path('/content/t1_brain_extraction_results')
archive_path = Path(shutil.make_archive(str(archive_base), 'zip', root_dir=RESULTS.parent, base_dir=RESULTS.name))
print(f'Downloading {archive_path} ({archive_path.stat().st_size / (1024**2):.1f} MB)')
files.download(str(archive_path))